# 프롬프트 버전 관리와 롤백

여러분이 LLM으로 들어오는 티켓을 알맞은 팀에 배정하는 제품 지원 시스템을 담당하는 PM이라고 상상해 보세요. API 관련 티켓이 플랫폼 팀으로 더 많이 가도록 라우팅 프롬프트를 손보고 싶습니다. 예전에는 프롬프트가 코드베이스 안에 있었습니다. 바꾸려면 PR, CI 실행, 배포가 필요했고 되돌리는 데도 같은 절차가 필요했습니다. Managed Agents는 프롬프트를 서버 측에 둡니다. `agents.update`를 호출할 때마다 변경 불가능한 새 버전이 만들어지고, 세션은 ID로 어떤 버전을 쓸지 고릅니다. 프롬프트 변경을 검토하고 승인하는 절차는 그대로지만, 코드 diff가 아니라 설정에 담긴 버전 번호를 승인하게 되고, 실행 중인 서비스는 재빌드 없이 변경을 반영합니다. 문제가 생기면 호출자를 예전 버전으로 되돌리기만 하면 롤백됩니다.

이 쿡북에서는 지원 티켓 분류 에이전트를 만들고, 시스템 프롬프트를 수정한 뒤, 성능이 떨어지면 롤백해 봅니다.

마치고 나면 다음을 하게 됩니다.
- 에이전트를 만들고 `version: 1`로 돌아오는 것을 확인
- 라벨이 붙은 테스트 세트로 특정 버전을 채점
- v2 프롬프트를 배포하고 버전 번호가 올라가는 것을 확인
- 성능이 떨어진 에이전트를 배포 없이 v1으로 롤백

## 사전 준비

**필수:**
- Python 3.11 이상
- `anthropic>=0.91.0`
- Anthropic API 키
- 이 디렉터리의 `example_data/prompt_versioning_and_rollback/support_tickets.jsonl` 픽스처

## 준비

In [1]:
%%capture
%pip install -q "anthropic>=0.91.0" python-dotenv

In [2]:
import json
import os
import time
from collections import defaultdict
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()
if not os.getenv("ANTHROPIC_API_KEY"):
    raise RuntimeError("Set ANTHROPIC_API_KEY in your environment or .env file")

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")
client = Anthropic()

## 에이전트 만들기 (버전 1)

먼저 환경(에이전트가 실행되는 컨테이너 템플릿)과 에이전트 자체를 만듭니다. 시스템 프롬프트는 짧습니다. 각 티켓을 팀과 우선순위로 분류하고 JSON만 응답하라는 내용입니다.

In [3]:
env = client.beta.environments.create(name="ticket-triage-env")
ENV_ID = env.id

V1_SYSTEM = """You are a support-ticket triage agent for a usage-billed API product.
Read the ticket and respond with ONLY a single line of raw JSON (no code fences, no prose):
{"team": "<billing|auth|api-platform|dashboard>", "priority": "<P1|P2|P3>"}
Route based on the customer's actual problem, not surface keywords."""

agent = client.beta.agents.create(name="ticket-triage", model=MODEL, system=V1_SYSTEM)
AGENT_ID = agent.id
print(f"env={ENV_ID}  agent={AGENT_ID} v{agent.version}")

env=env_017FKqTdU6dFgEe6zbbVQvE5  agent=agent_011CZo6t9Cr6Eg5TeFFPWPpi v1


`agents.create`가 우리가 요청하지 않았는데도 `version: 1`을 돌려줬습니다. 이후 수정할 때마다 같은 `id`에 새 버전이 만들어지며, 세션을 고정하고 롤백할 때 사용할 것이 바로 이 번호입니다.

## 라벨이 붙은 테스트 세트 불러오기

아래에는 팀마다 다섯 건씩, 총 스무 건의 티켓이 있고 올바른 배정이 이미 라벨로 붙어 있습니다. 채점의 기준이 될 정답입니다.

In [4]:
fixture = Path("example_data/prompt_versioning_and_rollback/support_tickets.jsonl")
tickets = [json.loads(line) for line in fixture.read_text().splitlines()]
teams = sorted({t["team"] for t in tickets})
print(f"{len(tickets)} tickets across {len(teams)} teams: {teams}")

20 tickets across 4 teams: ['api-platform', 'auth', 'billing', 'dashboard']


## 버전 1 채점하기

모든 티켓을 v1에 통과시켜 결과를 보겠습니다. 아래 `triage` 헬퍼는 특정 버전에 고정된 세션을 열고, 티켓 하나를 보내고, 세션이 유휴 상태가 될 때까지 `events.list`를 폴링한 뒤, `agent.message` 이벤트에서 JSON 판정을 파싱합니다.

이 쿡북에서 중요한 부분은 `sessions.create`의 `agent=` 인자입니다. 평범한 문자열(`agent=AGENT_ID`)을 넘기면 최신 버전이 무엇이든 그것을 쓰게 됩니다. `{"type": "agent", "id": ..., "version": ...}`을 넘기면 정확한 버전에 고정되는데, 통제된 비교에는 이쪽이 필요합니다.

In [5]:
def triage(version: int, ticket: dict) -> dict:
    """Run one ticket through a pinned agent version and return its verdict."""
    session = client.beta.sessions.create(
        agent={"type": "agent", "id": AGENT_ID, "version": version},
        environment_id=ENV_ID,
    )
    try:
        prompt = "Subject: " + ticket["subject"] + "\n\n" + ticket["body"]
        client.beta.sessions.events.send(
            session.id,
            events=[{"type": "user.message", "content": [{"type": "text", "text": prompt}]}],
        )
        deadline = time.time() + 60
        while time.time() < deadline:
            events = client.beta.sessions.events.list(session.id).data
            if events and events[-1].type == "session.status_idle":
                break
            time.sleep(1)
        else:
            raise TimeoutError(f"session {session.id} did not idle within 60s")
        agent_events = [e for e in events if e.type == "agent.message"]
        reply = "".join(b.text for e in agent_events for b in e.content)
        return json.loads(reply)
    finally:
        try:
            client.beta.sessions.archive(session.id)
        except Exception:  # noqa: S110
            pass


def score(version: int) -> dict:
    """Evaluate all tickets against the given version and return per-team accuracy."""
    hits = defaultdict(lambda: [0, 0])
    for t in tickets:
        pred = triage(version, t)
        hits[t["team"]][1] += 1
        if pred.get("team") == t["team"]:
            hits[t["team"]][0] += 1
    return dict(hits)


v1_scores = score(version=1)
print("v1 results:")
for team, (correct, total) in sorted(v1_scores.items()):
    print(f"  {team:14s} {correct}/{total}")

v1 results:
  api-platform   5/5
  auth           5/5
  billing        4/5
  dashboard      5/5


모델이 잘 해냈고 배정을 거의 다 맞혔습니다.

## 버전 2 배포하기

이제 PM이 변경을 배포합니다. API 사용량이나 요청 한도에 관한 것은 플랫폼 팀 소관이라고 에이전트에 알려 주는 라우팅 규칙입니다. `agents.update`로 에이전트를 새 시스템 프롬프트로 갱신합니다.

In [6]:
V2_SYSTEM = V1_SYSTEM + (
    "\n\nROUTING RULE: If the ticket text mentions API usage, rate limits, quotas, "
    "or request volume, route to api-platform. Apply this rule before any other "
    "consideration; do not second-guess it based on the rest of the ticket."
)
agent = client.beta.agents.update(AGENT_ID, version=agent.version, system=V2_SYSTEM)
print(f"agent {AGENT_ID} now at v{agent.version}")

agent agent_011CZo6t9Cr6Eg5TeFFPWPpi now at v2


### 코드 리뷰는 어디로 갔을까

방금 API 호출 한 번으로 에이전트를 바꿨습니다. 이 노트북에서는 아무 검토 없이 이뤄졌는데, 데모로는 괜찮지만 프로덕션 운영 방식은 아닙니다.

`agents.update`에는 내장 승인 워크플로가 없습니다. 워크스페이스의 어떤 키로도 호출할 수 있습니다. 호출자가 고정된 버전 대신 맨 에이전트 ID를 넘기고 있다면, 바로 다음 세션부터 새 프롬프트를 쓰게 됩니다. 배포가 필요 없는 대신 치르는 대가이며, 기능 플래그나 코드가 아니라 API로 관리하는 다른 설정에서도 마찬가지입니다.

검토 단계를 되돌려 놓는 패턴은 이렇습니다. 프로덕션 호출자는 항상 명시적인 버전에 고정하고, *그 고정된 번호*를 변경 관리 대상으로 삼습니다. 누구나 v2, v3, v10을 만들 수 있지만, 그 버전들은 트래픽 없이 서버에 놓여 있을 뿐입니다. 승격이란 프로덕션 호출자가 어떤 버전을 넘길지 알려 주는 설정을 갱신하는 것이고, 그 갱신은 평소의 검토 절차를 거칩니다. 이러면 버전을 만드는 일은 값싸지고 SDLC는 그대로 유지됩니다. 프로덕션 코드를 고치는 대신 설정 값만 바꾸면 모든 실행 주체가 변경을 반영하므로 여전히 이득입니다.

## 버전 2 채점하기

같은 평가를 v2에 고정해 수행합니다. 새 규칙은 범위가 넓고, 사용량 기반 과금 제품에서는 결제 티켓에서도 API 사용량 이야기가 나옵니다.

In [7]:
v2_scores = score(version=agent.version)

for team in sorted(v1_scores):
    c1, n1 = v1_scores[team]
    c2, n2 = v2_scores[team]
    flag = "  <-- regressed" if c2 < c1 else ""
    print(team)
    print(f"  v1: {c1}/{n1}")
    print(f"  v2: {c2}/{n2}{flag}")

api-platform
  v1: 5/5
  v2: 5/5
auth
  v1: 5/5
  v2: 5/5
billing
  v1: 4/5
  v2: 2/5  <-- regressed
dashboard
  v1: 5/5
  v2: 5/5


## 롤백하기

결제 분류가 나빠졌습니다. 버전 1은 여전히 서버에 있습니다. 롤백은 배포가 아닙니다. 호출자가 다시 `version: 1`을 넘기기만 하면 됩니다.

In [8]:
billing = [t for t in tickets if t["team"] == "billing"]
rerun = [triage(version=1, ticket=t).get("team") for t in billing]
print(f"billing tickets via version=1: {rerun.count('billing')}/{len(billing)} correct")

billing tickets via version=1: 4/5 correct


버전이 서버 측에 살아 있으므로, 프로덕션은 v1에 머무는 동안에도 PM이 v2를 계속 다듬거나 소량의 트래픽을 카나리로 흘려보낼 수 있습니다. 수정이 준비되면 v3를 만들어 같은 절차로 승격하면 됩니다.

## 정리

In [9]:
client.beta.agents.archive(AGENT_ID)
client.beta.environments.archive(ENV_ID)
print("archived")

archived


## 정리하며

여기서의 동작은 단순합니다(생성, 갱신, 고정, 재고정). 하지만 프롬프트가 버전이 매겨진 서버 측 리소스가 되면서, 위에서처럼 `update`를 실행하든 설정의 버전을 바꾸든 애플리케이션 코드와 무관하게 프롬프트를 평가하고 승격할 수 있게 됩니다.

여러분의 워크플로에 가져갈 만한 것 몇 가지:

- 프로덕션 호출자는 맨 에이전트 ID가 아니라 명시적인 버전에 고정하게 하세요. 승격하기 전까지 새 버전은 보이지 않습니다.
- 고정된 버전 번호를 프롬프트 변경의 관문으로 다루세요. 버전을 만드는 것은 탐색이고, 프로덕션 고정 값을 바꾸는 것이 검토를 거치는 일입니다.
- 위험이 큰 에이전트라면 트래픽의 일부를 새 버전으로 보내 비교한 뒤(이 노트북이 하는 것처럼) 완전히 승격하세요. 이 버전 관리를 사실상 기능 플래그처럼 쓸 수 있습니다.

### 다음 단계

- `client.beta.agents.versions.list(AGENT_ID)`는 에이전트의 모든 버전을 반환합니다.
- `managed_agents/`의 다른 노트북들은 세션, 커스텀 도구, 종단 간 패턴을 다룹니다.